# Google Analytics Performance Report

This notebook demonstrates the full siege_utilities GA4 reporting pipeline:

1. **Data Acquisition** — Real GA4 API with 1Password credentials, fallback to sample data
2. **KPI Dashboard** — Metric cards with period-over-period comparison
3. **Traffic Trends** — Time series visualization (daily users, sessions, pageviews)
4. **Traffic Sources** — Channel distribution with heatmapped tables
5. **Device Analysis** — Desktop vs mobile vs tablet breakdown
6. **Page Performance** — Top pages with heatmapped pageview intensity
7. **Geographic Analysis** — Top cities/regions by session volume
8. **Year-over-Year Analysis** — Longitudinal comparison (3 years)
9. **Performance Highlights** — Best/worst days and weeks
10. **Automated Insights** — Data-driven analysis and recommendations
11. **PDF Generation** — Professional branded multi-page report with headers/footers

### Architecture

This notebook uses:
- `GoogleAnalyticsConnector` + `CredentialManager` for live data
- `ChartGenerator` for all visualizations
- `ClientBrandingManager` for client-specific styling
- `generate_ga_report_pdf()` for the final PDF output

In [ ]:
# Imports
import sys
from datetime import datetime, timedelta
from pathlib import Path

# Add parent directory for imports if running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_14"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Branding configuration ---
from siege_utilities.reporting.client_branding import ClientBrandingManager
_branding_mgr = ClientBrandingManager()
BRANDING = _branding_mgr.get_client_branding('hillcrest')
COLORS = BRANDING['colors']

print(f"Output directory: {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})")
print(f"Branding: {BRANDING['name']} (primary={COLORS['primary']})")

## 1. Data Acquisition

Try to connect to the real GA4 API using `GoogleAnalyticsConnector` with 1Password
credentials. If credentials are unavailable, fall back to realistic sample data.

Both paths produce the same dict structure, so all downstream code works identically.

In [ ]:
from siege_utilities.reporting.examples.google_analytics_report_example import (
    generate_sample_ga_data,
    fetch_real_ga4_data,
    create_kpi_dashboard,
    create_traffic_trend_chart,
    create_traffic_sources_chart,
    create_heatmapped_table_style,
    generate_insights,
    generate_recommendations,
    generate_ga_report_pdf
)

# Configuration
GA4_PROPERTY_ID = "366963525"  # Hillcrest Children & Family Center
CLIENT_NAME = "Hillcrest Children & Family Center"
BRANDING_KEY = "hillcrest"

end_date = datetime.now()
start_date = end_date - timedelta(days=30)

# Try real GA4 API first, fallback to sample data
ga_data = fetch_real_ga4_data(
    property_id=GA4_PROPERTY_ID,
    start_date=start_date.strftime('%Y-%m-%d'),
    end_date=end_date.strftime('%Y-%m-%d'),
    vault='Private',
    account='Dheeraj_Chand_Family',
)

if ga_data:
    print("Using LIVE GA4 data from API")
else:
    print("GA4 credentials not available - using sample data")
    ga_data = generate_sample_ga_data(start_date, end_date)

print(f"Data Source: {ga_data.get('data_source', 'sample')}")
print(f"Date Range: {ga_data['date_range']['start']} to {ga_data['date_range']['end']}")
print(f"Total Users: {ga_data['totals']['users']:,}")
print(f"Total Sessions: {ga_data['totals']['sessions']:,}")
print(f"Avg Bounce Rate: {ga_data['totals']['avg_bounce_rate']:.1f}%")
print(f"Avg Session Duration: {ga_data['totals']['avg_session_duration']:.0f}s")
print(f"Period-over-Period: Users {ga_data['changes']['users']:+.1f}%, Sessions {ga_data['changes']['sessions']:+.1f}%")

## 2. KPI Dashboard

In [ ]:
# Display key metrics with period-over-period changes
totals = ga_data['totals']
changes = ga_data['changes']

metrics = {
    'Users': (f"{totals['users']:,}", f"{changes['users']:+.1f}%"),
    'Sessions': (f"{totals['sessions']:,}", f"{changes['sessions']:+.1f}%"),
    'Bounce Rate': (f"{totals['avg_bounce_rate']:.1f}%", f"{changes['bounce_rate']:+.1f}pp"),
    'Avg Duration': (f"{totals['avg_session_duration']:.0f}s", f"{changes['duration']:+.1f}%"),
    'Pages/Session': (f"{totals['pages_per_session']:.1f}", ""),
    'Pageviews': (f"{totals['pageviews']:,}", ""),
}

kpi_df = pd.DataFrame(
    [(k, v, c) for k, (v, c) in metrics.items()],
    columns=['Metric', 'Value', 'vs Prior Period']
)
print("KPI Summary:")
display(kpi_df)

## 3. Daily Traffic Trends

In [ ]:
# Create daily data DataFrame
daily_df = pd.DataFrame({
    'date': pd.to_datetime(ga_data['daily_data']['dates']),
    'users': ga_data['daily_data']['users'],
    'sessions': ga_data['daily_data']['sessions'],
    'pageviews': ga_data['daily_data']['pageviews'],
    'bounce_rate': ga_data['daily_data']['bounce_rate']
})

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Users and Sessions
ax1 = axes[0, 0]
ax1.plot(daily_df['date'], daily_df['users'], label='Users', color=COLORS['primary'], linewidth=2)
ax1.plot(daily_df['date'], daily_df['sessions'], label='Sessions', color=COLORS['secondary'], linewidth=2, alpha=0.7)
ax1.set_title('Daily Users & Sessions', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Pageviews
ax2 = axes[0, 1]
ax2.fill_between(daily_df['date'], daily_df['pageviews'], alpha=0.5, color=COLORS['secondary'])
ax2.plot(daily_df['date'], daily_df['pageviews'], color=COLORS['secondary'], linewidth=2)
ax2.set_title('Daily Pageviews', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# Bounce Rate
ax3 = axes[1, 0]
ax3.bar(daily_df['date'], daily_df['bounce_rate'], color=COLORS['accent'], alpha=0.7)
ax3.axhline(y=daily_df['bounce_rate'].mean(), color=COLORS['primary'], linestyle='--', label='Average')
ax3.set_title('Daily Bounce Rate', fontsize=12, fontweight='bold')
ax3.set_ylabel('Bounce Rate (%)')
ax3.legend()
ax3.tick_params(axis='x', rotation=45)

# Weekly pattern
ax4 = axes[1, 1]
daily_df['weekday'] = daily_df['date'].dt.day_name()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_avg = daily_df.groupby('weekday')['users'].mean().reindex(weekday_order)
ax4.bar(range(7), weekly_avg.values, color=COLORS['primary'])
ax4.set_xticks(range(7))
ax4.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
ax4.set_title('Average Users by Day of Week', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'traffic_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Traffic Sources Analysis

In [ ]:
# Traffic sources DataFrame
sources_df = pd.DataFrame(ga_data['traffic_sources'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart — use branding palette
chart_colors = [COLORS['primary'], COLORS['secondary'], COLORS['accent'], '#F39C12', '#8E44AD']
wedges, texts, autotexts = axes[0].pie(
    sources_df['sessions'],
    labels=sources_df['source'].str.title(),
    autopct='%1.1f%%',
    colors=chart_colors,
    startangle=90
)
axes[0].set_title('Traffic Sources Distribution', fontsize=12, fontweight='bold')

# Bar chart with bounce rate
x = range(len(sources_df))
width = 0.35
bars1 = axes[1].bar([i - width/2 for i in x], sources_df['sessions']/1000, width,
                     label='Sessions (K)', color=COLORS['primary'])
ax2 = axes[1].twinx()
bars2 = ax2.bar([i + width/2 for i in x], sources_df['bounce_rate'], width,
                label='Bounce Rate (%)', color=COLORS['accent'], alpha=0.7)

axes[1].set_xticks(x)
axes[1].set_xticklabels(sources_df['source'].str.title())
axes[1].set_ylabel('Sessions (thousands)', color=COLORS['primary'])
ax2.set_ylabel('Bounce Rate (%)', color=COLORS['accent'])
axes[1].set_title('Sessions vs Bounce Rate by Source', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'traffic_sources.png', dpi=150, bbox_inches='tight')
plt.show()

# Display table
print("\nTraffic Sources Summary:")
display_df = sources_df.copy()
display_df['sessions'] = display_df['sessions'].apply(lambda x: f"{x:,}")
display_df['users'] = display_df['users'].apply(lambda x: f"{x:,}")
display_df['bounce_rate'] = display_df['bounce_rate'].apply(lambda x: f"{x:.1f}%")
display_df['avg_duration'] = display_df['avg_duration'].apply(lambda x: f"{x:.1f}s")
display(display_df)

## 5. Top Pages Performance

In [ ]:
# Top pages DataFrame
pages_df = pd.DataFrame(ga_data['top_pages'])

fig, ax = plt.subplots(figsize=(12, 5))

# Horizontal bar chart with gradient
y_pos = range(len(pages_df))
bar_colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(pages_df)))
bars = ax.barh(y_pos, pages_df['pageviews'], color=bar_colors)

ax.set_yticks(y_pos)
ax.set_yticklabels(pages_df['page'])
ax.set_xlabel('Pageviews')
ax.set_title('Top Pages by Pageviews', fontsize=12, fontweight='bold')

# Add value labels
for bar, val in zip(bars, pages_df['pageviews']):
    ax.text(val + max(pages_df['pageviews'])*0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'top_pages.png', dpi=150, bbox_inches='tight')
plt.show()

# Display table
print("\nTop Pages Performance:")
display_pages = pages_df.copy()
display_pages['pageviews'] = display_pages['pageviews'].apply(lambda x: f"{x:,}")
display_pages['avg_time'] = display_pages['avg_time'].apply(lambda x: f"{x:.1f}s")
display_pages['bounce_rate'] = display_pages['bounce_rate'].apply(lambda x: f"{x:.1f}%")
display_pages['exit_rate'] = display_pages['exit_rate'].apply(lambda x: f"{x:.1f}%")
display(display_pages)

## 6. Device Analysis

In [ ]:
# Device breakdown
devices_df = pd.DataFrame(ga_data['devices'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors_dev = [COLORS['primary'], COLORS['secondary'], COLORS['accent']]
axes[0].pie(
    devices_df['sessions'],
    labels=devices_df['device'].str.title(),
    autopct='%1.1f%%',
    colors=colors_dev,
    startangle=90
)
axes[0].set_title('Sessions by Device', fontsize=12, fontweight='bold')

# Bounce rate comparison
bars = axes[1].bar(devices_df['device'].str.title(), devices_df['bounce_rate'], color=colors_dev)
axes[1].set_ylabel('Bounce Rate (%)')
axes[1].set_title('Bounce Rate by Device', fontsize=12, fontweight='bold')

for bar, val in zip(bars, devices_df['bounce_rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'device_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Geographic Distribution

In [ ]:
# Geographic data
geo_df = pd.DataFrame(ga_data['geo_data'])

fig, ax = plt.subplots(figsize=(10, 5))

bar_colors = plt.cm.Greens(np.linspace(0.4, 0.9, len(geo_df)))
bars = ax.barh(range(len(geo_df)), geo_df['sessions'], color=bar_colors)

ax.set_yticks(range(len(geo_df)))
ax.set_yticklabels([f"{row['city']}, {row['region']}" for _, row in geo_df.iterrows()])
ax.set_xlabel('Sessions')
ax.set_title('Top Cities by Sessions', fontsize=12, fontweight='bold')

for bar, val in zip(bars, geo_df['sessions']):
    ax.text(val + max(geo_df['sessions'])*0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'geographic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Year-over-Year Analysis

Longitudinal comparison across 3 years showing traffic growth trends.
Salvaged from Masai-Interactive/google_analytics_reports.

In [ ]:
longitudinal = ga_data.get('longitudinal', {})

if longitudinal:
    years = sorted(longitudinal.keys())
    sessions_by_year = [longitudinal[y]['sessions'] for y in years]
    users_by_year = [longitudinal[y]['users'] for y in years]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Sessions by year — use branding palette
    bars = axes[0].bar(years, sessions_by_year, color=[COLORS['primary'], COLORS['secondary'], COLORS['accent']])
    axes[0].set_title('Total Sessions by Year', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Sessions')
    for bar, val in zip(bars, sessions_by_year):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(sessions_by_year)*0.01,
                    f'{val:,}', ha='center', va='bottom', fontsize=9)

    # Growth rates
    growth_rates = [0]
    for i in range(1, len(sessions_by_year)):
        rate = ((sessions_by_year[i] - sessions_by_year[i-1]) / sessions_by_year[i-1]) * 100
        growth_rates.append(rate)

    bar_colors = [COLORS['secondary'] if r >= 0 else COLORS['accent'] for r in growth_rates]
    axes[1].bar(years, growth_rates, color=bar_colors)
    axes[1].set_title('Year-over-Year Growth Rate', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Growth (%)')
    axes[1].axhline(y=0, color='black', linewidth=0.5)
    for i, (bar_x, rate) in enumerate(zip(years, growth_rates)):
        label = 'Baseline' if i == 0 else f'{rate:+.1f}%'
        axes[1].text(i, rate + max(abs(r) for r in growth_rates)*0.05,
                    label, ha='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'yoy_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Display YoY table
    print("\nYear-over-Year Comparison:")
    for i, year in enumerate(years):
        data = longitudinal[year]
        growth = 'Baseline' if i == 0 else f'{growth_rates[i]:+.1f}%'
        print(f"  {year}: {data['sessions']:,} sessions, {data['users']:,} users  ({growth})")
else:
    print("No longitudinal data available")

## 9. Performance Highlights

Best and worst performing days and weeks.

In [ ]:
best_day = ga_data.get('best_day', {})
worst_day = ga_data.get('worst_day', {})
best_week = ga_data.get('best_week', {})
worst_week = ga_data.get('worst_week', {})

if best_day:
    print("Performance Highlights:")
    print(f"  Best Day:    {best_day['date']} - {best_day['sessions']:,} sessions")
    print(f"  Lowest Day:  {worst_day['date']} - {worst_day['sessions']:,} sessions")
    if best_week:
        print(f"  Best Week:   Week {best_week['week']} - {best_week['sessions']:,} sessions")
        print(f"  Lowest Week: Week {worst_week['week']} - {worst_week['sessions']:,} sessions")

    # Visualize best vs worst days in context
    daily_sessions = ga_data['daily_data']['sessions']
    daily_dates = pd.to_datetime(ga_data['daily_data']['dates'])

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(daily_dates, daily_sessions, color=COLORS['primary'], alpha=0.6)

    # Highlight best and worst
    best_idx = ga_data['daily_data']['dates'].index(best_day['date'])
    worst_idx = ga_data['daily_data']['dates'].index(worst_day['date'])
    ax.bar(daily_dates[best_idx], daily_sessions[best_idx], color=COLORS['secondary'], label='Best Day')
    ax.bar(daily_dates[worst_idx], daily_sessions[worst_idx], color=COLORS['accent'], label='Lowest Day')
    ax.axhline(y=np.mean(daily_sessions), color='#F39C12', linestyle='--', label='Average')
    ax.set_title('Daily Sessions with Best/Lowest Highlighted', fontsize=12, fontweight='bold')
    ax.legend()
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'performance_highlights.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Performance highlights not available")

## 10. Automated Insights & Recommendations

In [ ]:
# Generate insights
insights = generate_insights(ga_data)
recommendations = generate_recommendations(ga_data)

print("Key Insights:")
print("=" * 60)
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

print()
print("Recommendations:")
print("=" * 60)
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

## 11. Generate PDF Report

Produces a professional branded PDF with:
- Client-branded title page (Hillcrest/Siege Analytics colors)
- Table of contents
- KPI dashboard with metric cards
- All chart sections
- Heatmapped data tables (color-intensity by value)
- Year-over-year longitudinal analysis
- Best/worst day performance highlights
- Headers/footers on every page (client name, page numbers)

In [ ]:
output_path = str(OUTPUT_DIR / 'ga_performance_report.pdf')

success = generate_ga_report_pdf(
    ga_data=ga_data,
    output_path=output_path,
    client_name=CLIENT_NAME,
    report_title="Google Analytics Performance Report",
    branding_key=BRANDING_KEY,
    prepared_by="Masai Interactive / Siege Analytics"
)

if success:
    report_path = Path(output_path)
    print(f"Report generated: {output_path} ({report_path.stat().st_size / 1024:.0f} KB)")
    print("Report sections:")
    print("  - Branded Title Page")
    print("  - Table of Contents")
    print("  - Executive Summary + Key Performance Highlights")
    print("  - KPI Dashboard with metric cards")
    print("  - Traffic Trends chart")
    print("  - Traffic Sources (heatmapped table)")
    print("  - Device breakdown")
    print("  - Top Pages (heatmapped table)")
    print("  - Geographic distribution (heatmapped table)")
    if ga_data.get('longitudinal'):
        print("  - Year-over-Year Analysis (heatmapped table)")
    if ga_data.get('best_day'):
        print("  - Performance Highlights (best/worst days & weeks)")
    print("  - Key Insights")
    print("  - Recommendations")
    print("  - Headers/footers on every page")
else:
    print("ERROR: Report generation failed. Check logs for details.")

## 12. Production Usage

### Using Real GA4 Data

```python
from siege_utilities.analytics import GoogleAnalyticsConnector
from siege_utilities.config import get_google_service_account_from_1password

# Option 1: Service Account via 1Password (recommended)
ga = GoogleAnalyticsConnector(
    auth_method="service_account",
    service_account_data=get_google_service_account_from_1password()
)

# Option 2: Factory function
from siege_utilities.analytics import create_ga_connector_from_1password
ga = create_ga_connector_from_1password("Google Analytics Service Account")

# Fetch GA4 data
df = ga.get_ga4_data(
    property_id="366963525",
    start_date="2025-01-01",
    end_date="2025-01-31",
    metrics=["sessions", "totalUsers", "bounceRate", "averageSessionDuration"],
    dimensions=["date", "city", "deviceCategory"]
)
```

### Client Branding

```python
from siege_utilities.reporting import ClientBrandingManager

mgr = ClientBrandingManager()
print(mgr.list_clients())  # ['hillcrest', 'masai_interactive', 'siege_analytics']

# Create custom client branding
mgr.create_client_branding('my_client', {
    'name': 'My Client',
    'colors': {'primary': '#003366', 'secondary': '#006633', 'text_color': '#1A1A1A'},
    'fonts': {'default_font': 'Helvetica'},
})
```

### Geographic Analysis with Census Integration

```python
from siege_utilities.reporting.examples.ga_geographic_analysis import (
    geocode_ga_cities,
    aggregate_by_state,
    create_state_choropleth,
)

ga_df = geocode_ga_cities(ga_city_data)
state_df = aggregate_by_state(ga_df)
choropleth_path = create_state_choropleth(state_df, 'sessions')
```

## Summary

This notebook demonstrated the full siege_utilities GA4 reporting pipeline:

1. **Data Acquisition** — Real GA4 API with 1Password credentials, graceful fallback to sample data
2. **KPI Dashboard** — Period-over-period metric comparison
3. **Traffic Trends** — 4-panel visualization (users, sessions, pageviews, bounce rate, weekly pattern)
4. **Traffic Sources** — Channel distribution with pie chart + comparative bar chart
5. **Top Pages** — Horizontal bar chart with value labels + performance table
6. **Device Analysis** — Session distribution + bounce rate by device
7. **Geographic Distribution** — Top cities by session volume
8. **Year-over-Year Analysis** — Longitudinal comparison with growth rates
9. **Performance Highlights** — Best/worst days with contextual visualization
10. **Automated Insights** — Data-driven analysis and recommendations
11. **PDF Generation** — Professional branded report with heatmapped tables, headers/footers, TOC

### Key Architectural Patterns

- **Graceful degradation**: Real GA4 API → sample data (same dict structure)
- **Client branding**: `ClientBrandingManager` with predefined templates
- **Heatmapped tables**: `create_heatmapped_table_style()` for value-intensity coloring
- **1Password integration**: `CredentialManager` for secure credential retrieval